In [1]:
import pandas as pd
import numpy as np
import os
import json

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
FAQ_PATH = "../data/raw/faq.csv"

print("FAQ exists:", os.path.exists(FAQ_PATH))

FAQ exists: True


In [3]:
faq_df = pd.read_csv(FAQ_PATH)

print("FAQ shape:", faq_df.shape)
print("\nColumns:")
print(faq_df.columns.tolist())

print("\nFirst 5 records:")
display(faq_df.head())

FAQ shape: (150, 4)

Columns:
['faq_id', 'question', 'answer', 'category']

First 5 records:


,faq_id,question,answer,category
0,FAQ001,How do I track my order?,Go to My Orders > Select Order > Track Package...,Delivery Issue
1,FAQ002,What should I do if my order is delayed?,Check tracking first. If no update for 48 hour...,Delivery Issue
2,FAQ003,Can I change my delivery address after placing...,"Yes, before dispatch. Go to My Orders > Order ...",Delivery Issue
3,FAQ004,What happens if I miss my delivery?,Courier will reattempt delivery. You can also ...,Delivery Issue
4,FAQ005,How do I reschedule my delivery?,Use the tracking link in your SMS/email to sel...,Delivery Issue


In [4]:
print("=" * 60)
print("FAQ DATA VALIDATION")
print("=" * 60)

print("\nNumber of records:", len(faq_df))

print("\nMissing values:")
print(faq_df.isnull().sum())

print("\nDuplicate FAQ IDs:")
print(faq_df["faq_id"].duplicated().sum())

print("\nDuplicate questions:")
print(faq_df["question"].duplicated().sum())

print("\nCategories:")
print(faq_df["category"].value_counts())

FAQ DATA VALIDATION

Number of records: 150

Missing values:
faq_id      0
question    0
answer      0
category    0
dtype: int64

Duplicate FAQ IDs:
0

Duplicate questions:
0

Categories:
category
Product Issue               27
Account & Login             27
Payment Issue               24
Delivery Issue              22
Refund & Return             18
App & Website Issue         17
Seller & Product Listing    15
Name: count, dtype: int64


In [5]:
faq_df["document_text"] = (
    "Category: " + faq_df["category"].astype(str)
    + "\nQuestion: " + faq_df["question"].astype(str)
    + "\nAnswer: " + faq_df["answer"].astype(str)
)

print(faq_df["document_text"].iloc[0])

Category: Delivery Issue
Question: How do I track my order?
Answer: Go to My Orders > Select Order > Track Package for real-time courier updates.


In [6]:
for i in range(3):
    print("=" * 70)
    print(f"FAQ DOCUMENT {i + 1}")
    print("=" * 70)
    print(faq_df["document_text"].iloc[i])
    print()

FAQ DOCUMENT 1
Category: Delivery Issue
Question: How do I track my order?
Answer: Go to My Orders > Select Order > Track Package for real-time courier updates.

FAQ DOCUMENT 2
Category: Delivery Issue
Question: What should I do if my order is delayed?
Answer: Check tracking first. If no update for 48 hours, raise a complaint via My Orders > Help.

FAQ DOCUMENT 3
Category: Delivery Issue
Question: Can I change my delivery address after placing an order?
Answer: Yes, before dispatch. Go to My Orders > Order Details > Edit Address.



In [7]:
faq_df["document_length"] = faq_df["document_text"].str.len()

print(faq_df["document_length"].describe())

count    150.000000
mean     162.440000
std       14.179331
min      123.000000
25%      153.000000
50%      164.000000
75%      172.000000
max      197.000000
Name: document_length, dtype: float64


In [8]:
import sentence_transformers
import faiss

print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

e:\HCL Guvi\Multi-Agent Customer Support\Multi-Agent Customer Support Platform\.supportvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers: 6.1.0
FAISS: 1.15.1


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4494.49it/s]


Embedding model loaded successfully.


In [10]:
faq_documents = faq_df["document_text"].tolist()

faq_embeddings = embedding_model.encode(
    faq_documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:", faq_embeddings.shape)

Batches: 100%|██████████| 5/5 [00:00<00:00,  8.37it/s]

Embedding shape: (150, 384)


In [ ]:
import sentence_transformers
import faiss

print("sentence-transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)

e:\HCL Guvi\Multi-Agent Customer Support\Multi-Agent Customer Support Platform\.supportvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence-transformers: 6.1.0
FAISS: 1.15.1


In [11]:
print("Number of documents:", len(faq_documents))
print("Embedding dimensions:", faq_embeddings.shape[1])
print("Embedding dtype:", faq_embeddings.dtype)

Number of documents: 150
Embedding dimensions: 384
Embedding dtype: float32


In [12]:
import faiss

dimension = faq_embeddings.shape[1]

faq_index = faiss.IndexFlatIP(dimension)

faq_index.add(
    faq_embeddings.astype("float32")
)

print("FAISS index created successfully.")
print("Total vectors:", faq_index.ntotal)

FAISS index created successfully.
Total vectors: 150


In [13]:
def retrieve_faq(query, top_k=5):
    """
    Retrieve the most relevant FAQ documents for a query.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faq_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "faq_id": faq_df.iloc[idx]["faq_id"],
            "category": faq_df.iloc[idx]["category"],
            "question": faq_df.iloc[idx]["question"],
            "answer": faq_df.iloc[idx]["answer"],
            "score": float(score)
        })

    return pd.DataFrame(results)

In [14]:
query = "Where is my package?"

results = retrieve_faq(
    query,
    top_k=5
)

display(results)

,faq_id,category,question,answer,score
0,FAQ001,Delivery Issue,How do I track my order?,Go to My Orders > Select Order > Track Package...,0.695681
1,FAQ006,Delivery Issue,What if my package shows delivered but I didn'...,Raise a complaint within 48 hours via My Order...,0.692487
2,FAQ005,Delivery Issue,How do I reschedule my delivery?,Use the tracking link in your SMS/email to sel...,0.656978
3,FAQ002,Delivery Issue,What should I do if my order is delayed?,Check tracking first. If no update for 48 hour...,0.655020
4,FAQ047,Product Issue,What if I receive a wrong product?,Report via My Orders > Report Issue > Wrong It...,0.646869


In [15]:
test_queries = [
    "Where is my package?",
    "My payment was charged twice",
    "I want to return the product",
    "I cannot log into my account",
    "The app keeps crashing",
    "The product I received is damaged",
    "The seller description is incorrect"
]

for query in test_queries:

    print("=" * 80)
    print("QUERY:", query)

    results = retrieve_faq(
        query,
        top_k=3
    )

    for _, row in results.iterrows():
        print(
            f"\nScore: {row['score']:.4f}"
        )
        print(
            f"Category: {row['category']}"
        )
        print(
            f"Question: {row['question']}"
        )
        print(
            f"Answer: {row['answer']}"
        )

QUERY: Where is my package?

Score: 0.6957
Category: Delivery Issue
Question: How do I track my order?
Answer: Go to My Orders > Select Order > Track Package for real-time courier updates.

Score: 0.6925
Category: Delivery Issue
Question: What if my package shows delivered but I didn't receive it?
Answer: Raise a complaint within 48 hours via My Orders > Report > Item Not Received.

Score: 0.6570
Category: Delivery Issue
Question: How do I reschedule my delivery?
Answer: Use the tracking link in your SMS/email to select a new delivery slot.
QUERY: My payment was charged twice

Score: 0.6889
Category: Payment Issue
Question: Why was I charged more than the cart total?
Answer: Check for applicable taxes, delivery charges, or packaging fees added at checkout.

Score: 0.6876
Category: Payment Issue
Question: How to get a duplicate invoice?
Answer: Go to My Orders > Select Order > Download Invoice or contact support.

Score: 0.6413
Category: Payment Issue
Question: What should I do if my pa

In [16]:
correct_at_1 = 0
correct_at_3 = 0
correct_at_5 = 0

for i, query in enumerate(faq_df["question"]):

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faq_index.search(
        query_embedding,
        5
    )

    retrieved_indices = indices[0]

    if i == retrieved_indices[0]:
        correct_at_1 += 1

    if i in retrieved_indices[:3]:
        correct_at_3 += 1

    if i in retrieved_indices[:5]:
        correct_at_5 += 1


total = len(faq_df)

print("=" * 60)
print("FAQ RETRIEVAL EVALUATION")
print("=" * 60)

print(
    f"Recall@1: {correct_at_1 / total:.4f}"
)

print(
    f"Recall@3: {correct_at_3 / total:.4f}"
)

print(
    f"Recall@5: {correct_at_5 / total:.4f}"
)

FAQ RETRIEVAL EVALUATION
Recall@1: 0.9933
Recall@3: 1.0000
Recall@5: 1.0000


In [17]:
tickets_rag = pd.read_csv(
    "../data/raw/support_tickets.csv"
)

tickets_rag["clean_text"] = (
    tickets_rag["ticket_text"]
    .astype(str)
    .str.lower()
    .str.strip()
)

tickets_rag = tickets_rag.drop_duplicates(
    subset="clean_text"
).reset_index(drop=True)

print("RAG evaluation tickets:", len(tickets_rag))

RAG evaluation tickets: 8777


In [18]:
def evaluate_ticket_retrieval(
    tickets_df,
    top_k=5
):
    correct_at_1 = 0
    correct_at_3 = 0
    correct_at_5 = 0

    for _, row in tickets_df.iterrows():

        query = row["ticket_text"]
        actual_category = row["ticket_category"]

        query_embedding = embedding_model.encode(
            [query],
            normalize_embeddings=True
        ).astype("float32")

        scores, indices = faq_index.search(
            query_embedding,
            top_k
        )

        retrieved_categories = [
            faq_df.iloc[idx]["category"]
            for idx in indices[0]
        ]

        if retrieved_categories[0] == actual_category:
            correct_at_1 += 1

        if actual_category in retrieved_categories[:3]:
            correct_at_3 += 1

        if actual_category in retrieved_categories[:5]:
            correct_at_5 += 1

    total = len(tickets_df)

    return {
        "Recall@1": correct_at_1 / total,
        "Recall@3": correct_at_3 / total,
        "Recall@5": correct_at_5 / total
    }

In [19]:
ticket_rag_results = evaluate_ticket_retrieval(
    tickets_rag,
    top_k=5
)

print("=" * 60)
print("TICKET-BASED RAG EVALUATION")
print("=" * 60)

for metric, score in ticket_rag_results.items():
    print(f"{metric}: {score:.4f}")

TICKET-BASED RAG EVALUATION
Recall@1: 0.7091
Recall@3: 0.9179
Recall@5: 0.9593


In [20]:
VECTOR_STORE_DIR = "../data/vector_store"

os.makedirs(
    VECTOR_STORE_DIR,
    exist_ok=True
)

print("Vector store directory ready.")

Vector store directory ready.


In [21]:
faiss.write_index(
    faq_index,
    f"{VECTOR_STORE_DIR}/faq_index.faiss"
)

print("FAISS index saved successfully.")

FAISS index saved successfully.


In [22]:
faq_metadata = faq_df[
    [
        "faq_id",
        "question",
        "answer",
        "category"
    ]
].copy()

faq_metadata.to_csv(
    f"{VECTOR_STORE_DIR}/faq_metadata.csv",
    index=False
)

print("FAQ metadata saved successfully.")

FAQ metadata saved successfully.


In [23]:
loaded_index = faiss.read_index(
    f"{VECTOR_STORE_DIR}/faq_index.faiss"
)

loaded_metadata = pd.read_csv(
    f"{VECTOR_STORE_DIR}/faq_metadata.csv"
)

print("Loaded vectors:", loaded_index.ntotal)
print("Loaded metadata:", loaded_metadata.shape)

Loaded vectors: 150
Loaded metadata: (150, 4)


In [24]:
loaded_index = faiss.read_index(
    f"{VECTOR_STORE_DIR}/faq_index.faiss"
)

loaded_metadata = pd.read_csv(
    f"{VECTOR_STORE_DIR}/faq_metadata.csv"
)

print("Loaded vectors:", loaded_index.ntotal)
print("Loaded metadata:", loaded_metadata.shape)

Loaded vectors: 150
Loaded metadata: (150, 4)
